# Advertising Spend & Sales Prediction

This notebook analyzes the **advertising spend dataset**: 200 marketing campaigns with spend on TV, Radio, and Newspaper advertising, and the resulting product Sales.

It answers the questions from the dataset brief:
1. What is the average amount spent on TV advertising?
2. What is the correlation between radio advertising and sales?
3. Which advertising medium has the highest impact on sales?
4. Fit a linear regression on TV, Radio, Newspaper to predict Sales, and visualize predictions vs. actuals.
5. Predict sales for TV=$200, Radio=$40, Newspaper=$50.
6. How does performance change when the data is normalized?
7. What happens if only Radio and Newspaper are used as predictors?


## 1. Setup and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
# If running in Colab, upload advertising_sales_data.xlsx first:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_excel("advertising_sales_data.xlsx")
df.shape

## 2. Data cleaning

In [ ]:
df.info()
print()
print(df.isnull().sum())

Two campaigns are missing a `Radio` value. We impute with the column median, a reasonable choice given Radio's distribution isn't heavily skewed.

In [ ]:
print(df[df['Radio'].isnull()])
df['Radio'] = df['Radio'].fillna(df['Radio'].median())
df.describe()

## 3. Average TV advertising spend (Question 1)

In [ ]:
avg_tv = df['TV'].mean()
print(f"Average TV spend: ${avg_tv:.2f}")

## 4. Correlation between Radio and Sales (Question 2)

In [ ]:
corr_matrix = df[['TV','Radio','Newspaper','Sales']].corr()
corr_matrix.round(3)

In [ ]:
radio_sales_corr = df['Radio'].corr(df['Sales'])
print(f"Correlation between Radio spend and Sales: r = {radio_sales_corr:.3f}")

**Observation:** Radio has a moderate positive correlation with Sales (r ≈ 0.35) — more radio spend is associated with higher sales, but the relationship is far weaker than TV's.

## 5. Which medium has the highest impact? (Question 3)

In [ ]:
corrs = df[['TV','Radio','Newspaper']].corrwith(df['Sales']).sort_values(ascending=False)
corrs

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
colors = ['#065A82','#1C7293','#888780']
bars = ax.barh(corrs.index, corrs.values, color=colors)
for b, v in zip(bars, corrs.values):
    ax.text(v+0.02, b.get_y()+b.get_height()/2, f'{v:.2f}', va='center', fontweight='bold')
ax.set_xlabel("Correlation with Sales")
ax.set_xlim(0, 1.05)
plt.title("Correlation of each ad medium with Sales")
plt.tight_layout()
plt.show()

**TV advertising has, by far, the highest impact on sales** (r ≈ 0.90), followed by Radio (r ≈ 0.35) and Newspaper (r ≈ 0.16).

## 6. Linear regression: predicting Sales from all three media (Question 4)

In [ ]:
X = df[['TV','Radio','Newspaper']]
y = df['Sales']

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print("Coefficients:")
for col, coef in zip(X.columns, model.coef_):
    print(f"  {col}: {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}")
print(f"\nR²: {r2_score(y, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, y_pred)):.4f}")

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y, y_pred, alpha=0.6, color='#065A82', s=35, edgecolor='white', linewidth=0.5)
lims = [min(y.min(), y_pred.min())-1, max(y.max(), y_pred.max())+1]
plt.plot(lims, lims, color='#F94144', linewidth=2, linestyle='--', label='Perfect prediction')
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title(f"Actual vs. Predicted Sales (R² = {r2_score(y, y_pred):.3f})")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

**Observation:** The model explains about 90% of the variance in sales (R² ≈ 0.90) and predictions track actuals closely, clustering tightly around the diagonal.

### Visualizing the dominant relationship: TV spend vs. Sales

In [ ]:
plt.figure(figsize=(7.5,5))
plt.scatter(df['TV'], df['Sales'], alpha=0.5, color='#065A82', s=30)
z = np.polyfit(df['TV'], df['Sales'], 1)
xs = np.linspace(df['TV'].min(), df['TV'].max(), 100)
plt.plot(xs, np.polyval(z, xs), color='#F94144', linewidth=2.5)
plt.xlabel("TV advertising spend ($)")
plt.ylabel("Sales (units)")
plt.title("TV spend vs. Sales with fitted trend line")
plt.tight_layout()
plt.show()

## 7. Predicting sales for new spend levels (Question 5)

In [ ]:
new_data = pd.DataFrame({'TV':[200], 'Radio':[40], 'Newspaper':[50]})
predicted_sales = model.predict(new_data)[0]
print(f"Predicted sales for TV=$200, Radio=$40, Newspaper=$50: {predicted_sales:.2f} units")

## 8. Effect of normalization on model performance (Question 6)

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_norm = scaler_X.fit_transform(X)
y_norm = scaler_y.fit_transform(y.values.reshape(-1,1)).ravel()

model_norm = LinearRegression()
model_norm.fit(X_norm, y_norm)
y_norm_pred = model_norm.predict(X_norm)

print(f"R² with normalized data:   {r2_score(y_norm, y_norm_pred):.4f}")
print(f"R² with unnormalized data: {r2_score(y, y_pred):.4f}")

**Observation:** R² is identical whether or not the data is normalized. This is expected — ordinary least squares regression is scale-invariant in its fit quality; normalization rescales the *coefficients* (making them comparable across variables) but doesn't change how well the line fits the data or the resulting predictions once converted back to the original scale.

Where normalization *does* help is in directly comparing each medium's relative impact, since raw coefficients are not comparable across variables with different scales (e.g. TV spend ranges 0–296 vs. Newspaper 0–114):

In [ ]:
model_std = LinearRegression()
model_std.fit(X_norm, y)  # standardized inputs, original-scale target
std_coefs = pd.Series(model_std.coef_, index=X.columns).sort_values(ascending=False)
std_coefs

With inputs standardized, the coefficients confirm TV has by far the largest standardized effect on sales, followed by Radio, with Newspaper contributing almost nothing.

## 9. Model using only Radio and Newspaper (Question 7)

In [ ]:
X_rn = df[['Radio','Newspaper']]
model_rn = LinearRegression()
model_rn.fit(X_rn, y)
y_rn_pred = model_rn.predict(X_rn)

print("Coefficients:")
for col, coef in zip(X_rn.columns, model_rn.coef_):
    print(f"  {col}: {coef:.4f}")
print(f"Intercept: {model_rn.intercept_:.4f}")
print(f"\nR²: {r2_score(y, y_rn_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, y_rn_pred)):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5,4.8))
models_names = ['Full model\n(TV+Radio+News)', 'Radio + Newspaper\nonly']
r2s = [r2_score(y, y_pred), r2_score(y, y_rn_pred)]
bars = ax.bar(models_names, r2s, color=['#065A82','#F94144'], width=0.5)
for b, v in zip(bars, r2s):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.3f}', ha='center', fontweight='bold')
ax.set_ylabel("R² (variance explained)")
ax.set_ylim(0, 1.05)
plt.title("Model performance: with vs. without TV")
plt.tight_layout()
plt.show()

**Observation:** Removing TV from the model causes R² to collapse from ~0.90 to ~0.12 — TV is doing the overwhelming majority of the predictive work. Radio and Newspaper alone barely explain sales variance, confirming TV advertising is the dominant driver in this dataset.

## 10. Summary of findings

1. **Average TV spend** across the 200 campaigns is about $147.
2. **Radio correlates moderately with Sales** (r ≈ 0.35) — a real but secondary relationship.
3. **TV advertising has by far the highest impact on Sales** (r ≈ 0.90, and the largest standardized regression coefficient).
4. **The full 3-variable linear model fits very well** (R² ≈ 0.90), with predictions tracking actual sales closely.
5. **Predicted sales for TV=$200, Radio=$40, Newspaper=$50** is about **19.8 units**.
6. **Normalization doesn't change model fit (R²)** — OLS is scale-invariant — but it does make coefficients comparable across predictors, which is useful for ranking impact.
7. **Dropping TV is costly**: a Radio+Newspaper-only model's R² falls to ~0.12, confirming Newspaper contributes almost nothing and TV is essential to the model's predictive power.
